In [1]:
import pandas as pd

results = pd.read_csv("../data/results.csv")
drivers = pd.read_csv("../data/drivers.csv")
constructors = pd.read_csv("../data/constructors.csv")
races = pd.read_csv("../data/races.csv")
circuits = pd.read_csv("../data/circuits.csv")
qualifying = pd.read_csv("../data/qualifying.csv")
pit_stops = pd.read_csv("../data/pit_stops.csv")

In [2]:
drivers = drivers.drop(columns=['url'], errors='ignore')
constructors = constructors.drop(columns=['url'], errors='ignore')
races = races.drop(columns=['url'], errors='ignore')
circuits = circuits.drop(columns=['url'], errors='ignore')

In [3]:
pit_summary = pit_stops.groupby(['raceId', 'driverId']).agg(
    pit_stop_count=('stop', 'count'),
    avg_pit_duration_ms=('milliseconds', 'mean'),
    total_pit_duration_ms=('milliseconds', 'sum')
).reset_index()

pit_summary.head()

,raceId,driverId,pit_stop_count,avg_pit_duration_ms,total_pit_duration_ms
0,841,1,2,23213.0,46426
1,841,2,2,24046.0,48092
2,841,3,1,23716.0,23716
3,841,4,3,24055.0,72165
4,841,5,1,24865.0,24865


In [4]:
# Convert ms → seconds
pit_summary['avg_pit_duration_sec'] = pit_summary['avg_pit_duration_ms'] / 1000
pit_summary['total_pit_duration_sec'] = pit_summary['total_pit_duration_ms'] / 1000

# Optional: round for clean display
pit_summary['avg_pit_duration_sec'] = pit_summary['avg_pit_duration_sec'].round(2)
pit_summary['total_pit_duration_sec'] = pit_summary['total_pit_duration_sec'].round(2)

pit_summary.head()

,raceId,driverId,pit_stop_count,avg_pit_duration_ms,total_pit_duration_ms,avg_pit_duration_sec,total_pit_duration_sec
0,841,1,2,23213.0,46426,23.21,46.43
1,841,2,2,24046.0,48092,24.05,48.09
2,841,3,1,23716.0,23716,23.72,23.72
3,841,4,3,24055.0,72165,24.06,72.17
4,841,5,1,24865.0,24865,24.86,24.86


In [5]:
# Keep only useful qualifying columns
qualifying_clean = qualifying[['raceId', 'driverId', 'constructorId', 'position']].copy()

In [6]:
# Rename qualifying position so it doesn't conflict with race finishing position
qualifying_clean = qualifying_clean.rename(columns={
    'position': 'qualifying_position'
})

In [7]:
# Convert qualifying position to numeric
qualifying_clean['qualifying_position'] = pd.to_numeric(
    qualifying_clean['qualifying_position'], 
    errors='coerce'
)

qualifying_clean.head()

,raceId,driverId,constructorId,qualifying_position
0,18,1,1,1
1,18,9,2,2
2,18,5,1,3
3,18,13,6,4
4,18,2,2,5


In [8]:
qualifying_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10494 entries, 0 to 10493
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   raceId               10494 non-null  int64
 1   driverId             10494 non-null  int64
 2   constructorId        10494 non-null  int64
 3   qualifying_position  10494 non-null  int64
dtypes: int64(4)
memory usage: 328.1 KB


In [9]:
strategy_df = results.merge(drivers, on='driverId', how='left') \
                    .merge(races, on='raceId', how='left') \
                    .merge(constructors, on='constructorId', how='left') \
                    .merge(circuits, on='circuitId', how='left') \
                    .merge(qualifying_clean[['raceId', 'driverId', 'qualifying_position']],
                           on=['raceId', 'driverId'], how='left') \
                    .merge(pit_summary, on=['raceId', 'driverId'], how='left')

strategy_df.head()

,resultId,raceId,driverId,constructorId,number_x,grid,position,positionText,positionOrder,points,...,country,lat,lng,alt,qualifying_position,pit_stop_count,avg_pit_duration_ms,total_pit_duration_ms,avg_pit_duration_sec,total_pit_duration_sec
0,1,18,1,1,22,1,1,1,1,10.0,...,Australia,-37.8497,144.968,10,1.0,NaN,NaN,NaN,NaN,NaN
1,2,18,2,2,3,5,2,2,2,8.0,...,Australia,-37.8497,144.968,10,5.0,NaN,NaN,NaN,NaN,NaN
2,3,18,3,3,7,7,3,3,3,6.0,...,Australia,-37.8497,144.968,10,7.0,NaN,NaN,NaN,NaN,NaN
3,4,18,4,4,5,11,4,4,4,5.0,...,Australia,-37.8497,144.968,10,12.0,NaN,NaN,NaN,NaN,NaN
4,5,18,5,1,23,3,5,5,5,4.0,...,Australia,-37.8497,144.968,10,3.0,NaN,NaN,NaN,NaN,NaN


In [10]:
strategy_df.shape

(26759, 57)

In [11]:
strategy_df[['driverId', 'raceId', 'qualifying_position', 'pit_stop_count']].head()

,driverId,raceId,qualifying_position,pit_stop_count
0,1,18,1.0,NaN
1,2,18,5.0,NaN
2,3,18,7.0,NaN
3,4,18,12.0,NaN
4,5,18,3.0,NaN
